In [ ]:
import pymupdf4llm as pdf
import pymupdf
from pathlib import Path


doc_dir = Path(r'./documents/').glob('*')
ex_dir = Path('./extracted_documents/'); ex_dir.mkdir(exist_ok=True)

for doc_path in doc_dir:
    doc = pymupdf.open(doc_path)

    for i in range(len(doc)):
        doc_md = pdf.to_markdown(doc,pages=[i])
        out = ex_dir.joinpath(Path(f'{doc_path.stem}')); out.mkdir(parents=True,exist_ok=True)

        (out / f"{i+1:03}.md").write_text(doc_md, encoding="utf-8")




In [12]:
import os
from dotenv import load_dotenv

load_dotenv()
INFOSYS_PATH = os.getenv('INFOSYS_PATH')
DATABASE_PATH = os.getenv('DATABASE_PATH')

Path(INFOSYS_PATH).is_dir()

True

In [ ]:
doc =  pdf.to_markdown(Path(r'documents\infosys_Integrated-annual-report-2025-26.pdf'),pages=[98])
print(doc)

In [ ]:
import pymupdf 

d = pymupdf.open(Path(r'documents\infosys_Integrated-annual-report-2025-26.pdf'))

page = d[96]

print(len(page.get_images()), "images")
print(len(page.get_drawings()), "vector drawings")
print(page.get_text()[:800])

In [1]:
import re
from pathlib import Path

def chunk_page(text, page, max_chars=2000):
    # split into blocks at blank lines; a table stays one block
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    chunks, cur = [], ""
    for b in blocks:
        if cur and len(cur) + len(b) > max_chars:
            chunks.append("search_document: " + cur); cur = b
        else:
            cur = f"{cur}\n\n{b}" if cur else b
    if cur: chunks.append("search_document: " + cur)
    return [{"text": c, "page": page} for c in chunks]

all_chunks = []
for f in sorted(Path("cleaned_documents/infosys_Integrated-annual-report-2025-26").glob("*.md")):
    all_chunks += chunk_page(f.read_text(encoding="utf-8"), int(f.stem))

# Path('./chunks.txt').write_bytes(all_chunks)
print(len(all_chunks))

890


In [2]:
all_chunks[0:3]

[{'text': 'search_document: # From possibilities to enterprise AI impact\n\nThere’s a particular moment, in every technology shift, when the conversation changes. The question stops being whether the technology is real, or whether it matters, or whether the enterprise should pay attention. Those questions get answered. What replaces them is harder, and more honest. Where is the value? Who is realizing it? And what does it take to claim it at scale? That moment, for AI, is now.\n\nAcross boardrooms and operating reviews, the conversation has turned from possibilities to outcomes. Enterprises that began with curiosity and pilots are asking sharper questions. How does AI move the needle on revenue, on cost, on customer experience, on the speed at which the business can respond to change? How does it stop being an experiment run by an enthusiastic team and become the operating substrate of the enterprise: the way decisions get made, the way work gets done, the way trust gets built with cus

In [10]:
test_chunk = all_chunks[:3]

In [3]:
import ollama, chromadb

db_path = Path("./chroma_db")
db_path.mkdir(exist_ok=True)
client = chromadb.PersistentClient(path=db_path)
collection  = client.get_or_create_collection(name='my_docs')

ids = []
embeddings = []
metadatas = []
org_doc = []
for i,chunk in enumerate(all_chunks,start=1):
    response = ollama.embeddings(model='nomic-embed-text',prompt=f"""{chunk['text']}""")

    embeddings.append(response.embedding)
    ids.append(f'chunk_{i}')
    metadatas.append({'page':chunk['page']})
    org_doc.append(chunk['text'])


collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=org_doc,
    metadatas=metadatas
)




In [3]:
collection.count()

890

In [6]:

recent = 'help'
query = 'jail'
messages=[{'role':'user','content':                
        f"""Chat so far:\n{recent}\n\nRewrite the last question as a standalone question.
        Output only the question.\nQuestion: {query}"""}]

messages

[{'role': 'user',
  'content': 'Chat so far:\nhelp\n\nRewrite the last question as a standalone question.\n        Output only the question.\nQuestion: jail'}]

In [7]:
query = 'Tell me about infosys'

query_res = ollama.embeddings(model='nomic-embed-text',prompt=f"{query}")

query_emb = query_res.embedding

answer = collection.query(query_embeddings=[query_emb],n_results=2)



In [8]:
answer

{'ids': [['chunk_18', 'chunk_178']],
 'embeddings': None,
 'documents': [['search_document: # **Corporate overview**\n\n## About Infosys\n\nInfosys is a global leader in AI-first Since we refreshed our ESG Vision business consulting and technology 2030 in 2025, we have translated services. Over 3,25,000 of our ambition into action: we have people work to amplify human advanced our Climate Positive goals, potential and create the next scaled our skilling and community opportunity for people, businesses, initiatives, and strengthened our and communities. As navigators of inclusive workforce. Our governance enterprise transformation, we enable and ethics earn global recognition, businesses in 59 countries to unlock with accountability that extends AI value at scale. across our value chain. What began as a renewed commitment has With over four decades of experience become demonstrable progress in managing the systems and toward a sustainable future for all.\n\nWith over four decades of exp

In [10]:
answer['metadatas'][0]

[{'page': 10}, {'page': 75}]

In [14]:
import time
while True:
    print('While',end='')
    time.sleep(1);print('.',end='');time.sleep(1);print('.',end='');time.sleep(1);print('.\r',end='')
    

While.........

KeyboardInterrupt: 

In [15]:
import time

for i in range(1, 6):
    print(f"Progress: {i}/5", flush=True)
    time.sleep(1)
    if i < 5:
        print("\033[F\033[K", end="")


Progress: 1/5
Progress: 2/5
Progress: 3/5
Progress: 4/5
Progress: 5/5
